# <center>Organización de Datos</center>
#### <center>Cátedra Ing. Rodriguez, Juan Manuel </center>

## <center>Feature Engineering</center>
### <center>Práctica Análisis de Valores Atípicos</center>

> **Versión corregida y ampliada.** Cambios respecto de la original:
> - Se corrigieron los límites de outliers por IQR (estaban usando Q1 en vez de Q3 y viceversa en dos de los tres casos).
> - Se corrigió una comparación que mezclaba `z_edad` con `zm_edad`.
> - El umbral de Mahalanobis ahora se deriva de la distribución Chi-cuadrado en vez de fijarse "a ojo".
> - Se agregó el umbral faltante para LOF (estaba pedido como TODO y sin resolver).
> - Se agregó una mención a estimadores robustos de covarianza (MCD) para el problema de *masking*.
> - Se agregó una tabla comparativa final de los métodos vistos.

En esta notebook vamos a practicar las técnicas para detectar outliers univariados y multivariados.

## Carga de librerías y dataset

In [ ]:
import pandas as pd
import numpy as np
import sklearn as sk

import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as st
import scipy.linalg as la
from scipy.spatial.distance import cdist

import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

El dataset contiene información sobre la edad, altura y peso de 28 personas adultas de sexo masculino.

In [ ]:
df = pd.read_csv("/content/varones_adultos.csv")
df_base = df.copy(deep=True)
df.head()

## Análisis univariado

Mediante el análisis de los gráficos de tipo Box-Plot vamos a intentar identificar outliers univariados.

Recordatorio de la regla de Tukey: dado el rango intercuartil $IQR = Q3 - Q1$,

$$\text{límite inferior} = Q1 - 1.5 \cdot IQR \qquad \text{límite superior} = Q3 + 1.5 \cdot IQR$$

**TODO:** Para nuestros datos ¿cuál sería el valor de Edad a partir del cual una observación se consideraría outlier superior?

**TODO:** ¿Existen outliers severos? (pista: la regla "severa" usa 3 en vez de 1.5)

In [ ]:
sns.boxplot(y=df.Altura)
plt.show()

# Cuartiles
Q1_altura = np.quantile(df.Altura, 0.25)
Q3_altura = np.quantile(df.Altura, 0.75)

# Rango intercuartil
IQR_altura = Q3_altura - Q1_altura

# Límite inferior de outliers (Q1 - 1.5*IQR)
out_inf = Q1_altura - 1.5 * IQR_altura

# Límite superior de outliers (Q3 + 1.5*IQR)
out_sup = Q3_altura + 1.5 * IQR_altura

out_inf, out_sup

In [ ]:
sns.boxplot(y=df.Peso)
plt.show()

# Cuartiles
Q1_peso = np.quantile(df.Peso, 0.25)
Q3_peso = np.quantile(df.Peso, 0.75)

# Rango intercuartil
IQR_peso = Q3_peso - Q1_peso

# Límite inferior de outliers (Q1 - 1.5*IQR)
out_inf = Q1_peso - 1.5 * IQR_peso

# Límite superior de outliers (Q3 + 1.5*IQR)
out_sup = Q3_peso + 1.5 * IQR_peso

out_inf, out_sup

In [ ]:
sns.boxplot(y=df.Edad)
plt.show()

Encontramos que sólo existen outliers univariados evidentes en la variable Edad.

Verificamos que no se trata de un valor posible y procedemos a eliminarlo.

In [ ]:
# Observación anómala
outlier = df[df["Edad"] > 100]
outlier

In [ ]:
# Eliminación de outlier por índice de fila
indice_outlier = df[df["Edad"] > 100].index
df.drop(indice_outlier, inplace=True)

Visualicemos nuevamente el Box-Plot.

In [ ]:
sns.boxplot(y=df.Edad)
plt.show()

# Cuartiles
Q1_edad = np.quantile(df.Edad, 0.25)
Q3_edad = np.quantile(df.Edad, 0.75)

# Rango intercuartil
IQR_edad = Q3_edad - Q1_edad

# Límite inferior de outliers (Q1 - 1.5*IQR)
out_inf = Q1_edad - 1.5 * IQR_edad

# Límite superior de outliers (Q3 + 1.5*IQR)
out_sup = Q3_edad + 1.5 * IQR_edad

out_inf, out_sup

**¿Debemos eliminar las nuevas observaciones atípicas?**

No necesariamente: a diferencia de la Edad > 100 (un valor imposible), estas nuevas observaciones son valores plausibles que simplemente están en la cola de la distribución. Antes de eliminar hay que preguntarse si son errores de carga/medición o si son variabilidad real de la población.

### Z-Score, normal y modificado

Ahora probemos las técnicas z-score y z-score modificado para las variables del dataset.

**TODO:** ¿Se detectan anomalías que no se visualizaron en el boxplot?

**TODO:** Calcular los scores para la variable Altura.

In [ ]:
def z_calculation(serie):
    media = np.mean(serie)
    std = np.std(serie)
    return (serie - media) / std

# Z-score Edad
df["z_edad"] = z_calculation(df.Edad)

# Z-score Peso con librería stats
df["z_peso"] = st.zscore(df.Peso)

In [ ]:
def z_mod_calculation(serie):
    median = np.median(serie)
    MAD = np.median(np.absolute(serie - median))
    return (serie - median) * 0.6745 / MAD

# Z-score modificado Edad
df["zm_edad"] = z_mod_calculation(df.Edad)

# Z-score modificado Peso
df["zm_peso"] = z_mod_calculation(df.Peso)

df.head()

Verifiquemos si se cumple la "regla de oro" (|z| > 3 para z-score normal, |z_mod| > 3.5 para el modificado) para el caso de la variable Edad.

**TODO:** Verificar para el resto de las variables (Peso).

In [ ]:
df[df["z_edad"] > 3]

In [ ]:
df[df["z_edad"] < -3]

In [ ]:
plt.hist(df.z_edad)
plt.title("Histograma Z-Score Edad")
plt.xlabel("Z-Score Edad")
plt.show()

In [ ]:
df[df["zm_edad"] > 3.5]

In [ ]:
df[df["zm_edad"] < -3.5]

In [ ]:
plt.hist(df.zm_edad)
plt.title("Histograma Z-Score Modificado Edad")
plt.xlabel("ZM-Score Edad")
plt.show()

Cargamos el dataset base (con el outlier de Edad todavía adentro) y vemos qué hubiéramos detectado *antes* de removerlo.

In [ ]:
df_zs_base = df_base.copy(deep=True)

# Z-score y z-score modificado sobre Edad, sin remover el outlier
df_zs_base["z_edad"] = z_calculation(df_zs_base.Edad)
df_zs_base["zm_edad"] = z_mod_calculation(df_zs_base.Edad)
df_zs_base.head()

In [ ]:
# Regla de oro z-score normal (|z| > 3)
df_zs_base[(df_zs_base["z_edad"] < -3) | (df_zs_base["z_edad"] > 3)]

In [ ]:
# Regla de oro z-score modificado (|z_mod| > 3.5)
# Nota: en la versión original esta celda comparaba por error "z_edad" en vez de "zm_edad"
# en el límite superior. Acá quedan ambos lados evaluados sobre la misma variable.
df_zs_base[(df_zs_base["zm_edad"] < -3.5) | (df_zs_base["zm_edad"] > 3.5)]

Noten que el z-score normal es mucho más sensible al outlier extremo que el z-score modificado: la media y el desvío estándar (usados por z-score) se distorsionan fuertemente con un solo valor extremo, mientras que la mediana y el MAD (usados por z-score modificado) son robustos a eso. Por eso, en presencia de outliers severos, el z-score modificado suele ser más confiable para *detectarlos* (no se "esconden" a sí mismos).

## Análisis Multivariado - Mahalanobis

Vamos a analizar la presencia de outliers multivariados.
Exploremos las variables Peso y Altura.

**TODO:** ¿Existen outliers multivariados entre Edad y Peso? ¿Entre Edad y Altura?

In [ ]:
# Scatter Plot
plt.scatter(df.Peso, df.Altura)
plt.title("Dispersograma Peso vs Altura")
plt.xlabel("Peso")
plt.ylabel("Altura")
plt.show()

Parecerían existir algunas observaciones anómalas, calculemos la distancia de Mahalanobis para cada observación.

In [ ]:
def mahalanobis_distance(
    df,
    cols,
    sample_frac=1.0,
    squared=False,
    return_params=False
):
    """
    Calcula la distancia de Mahalanobis para cada observación.

    Parámetros
    ----------
    df           : DataFrame con los datos
    cols         : lista de columnas a usar, ej. ["Peso", "Altura"]
    sample_frac  : fracción para estimar mu y Sigma (default: 1.0)
    squared      : si True retorna D², si False retorna D (default: False)
    return_params : si True retorna también mu y cov estimados

    Retorna
    -------
    DataFrame con columna 'mahalanobis' agregada (y opcionalmente mu, cov)
    """
    X = df[cols].values.astype(float)
    sample = df[cols].sample(frac=sample_frac).values.astype(float)

    # Estimación de mu y Sigma desde la muestra
    mu = sample.mean(axis=0)
    cov = np.cov(sample.T)  # usa n-1 (insesgado)

    try:
        # Descomposición de Cholesky: cov = L @ L.T, con L triangular inferior.
        # Más estable numéricamente que invertir cov directamente,
        # y evita valores complejos que puede producir sqrtm().
        L = np.linalg.cholesky(cov)
        L_inv = np.linalg.inv(L)
    except np.linalg.LinAlgError:
        # La matriz de cov no es definida positiva: ocurre cuando hay features
        # colineales o constantes. Se regulariza sumando eps*I a la diagonal,
        # lo que la "empuja" a ser def. positiva sin alterar significativamente
        # la geometría de los datos.
        cov_reg = cov + np.eye(len(cols)) * 1e-8
        L = np.linalg.cholesky(cov_reg)
        L_inv = np.linalg.inv(L)

    # Usando la identidad: Sigma^-1 = L^-T L^-1
    # D^2(x) = (x-mu)^T Sigma^-1 (x-mu) = || L^-1 (x-mu) ||^2
    # Se evita calcular Sigma^-1 explícitamente; solo se transforma x-mu con L^-1.
    X_centered = X - mu
    Wx = X_centered @ L_inv.T
    d2 = np.sum(Wx ** 2, axis=1)

    result = df.copy()
    result["mahalanobis"] = np.sqrt(d2) if not squared else d2

    if return_params:
        return result, {"mu": mu, "cov": cov}
    return result

In [ ]:
df = mahalanobis_distance(df, cols=["Peso", "Altura"])
df.head()

Tenemos que seleccionar un valor umbral para definir qué observaciones podrían ser anómalas según la distancia de Mahalanobis.

### ¿De dónde sale el umbral? (esto faltaba en la versión original)

Si las variables provienen de una distribución normal multivariada, el cuadrado de la distancia de Mahalanobis, $D^2$, se distribuye aproximadamente como una **Chi-cuadrado con $p$ grados de libertad**, donde $p$ es la cantidad de variables usadas. Eso nos da una forma de elegir el umbral con fundamento estadístico en vez de "probar valores a ojo":

$$\text{umbral} = \sqrt{\chi^2_{p,\, 1-\alpha}}$$

Con $\alpha = 0.025$ (equivalente, aproximadamente, a la regla de "3 desvíos" en una dimensión) y $p=2$ variables:

In [ ]:
sns.boxplot(y=df.mahalanobis)
plt.show()

# Ordeno las distancias de menor a mayor
np.sort(df.mahalanobis)

In [ ]:
from scipy.stats import chi2

p = 2  # cantidad de variables: Peso, Altura
alpha = 0.025

# Umbral con base estadística: sqrt del cuantil chi-cuadrado
umbral = np.sqrt(chi2.ppf(1 - alpha, df=p))
umbral

In [ ]:
# Observaciones anómalas
df[df["mahalanobis"] > umbral]

In [ ]:
# Gráfico scatter
es_outlier = df["mahalanobis"] > umbral

sns.scatterplot(x=df.Altura, y=df.Peso, hue=es_outlier)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", borderaxespad=0, title="Outlier")
plt.title("Dispersograma Peso vs Altura")
plt.show()

Consideremos ahora las tres variables Edad, Altura y Peso. Calculemos nuevamente Mahalanobis — y noten que el umbral cambia, porque ahora $p=3$ (más grados de libertad → el umbral crece).

In [ ]:
df = mahalanobis_distance(df, cols=["Edad", "Peso", "Altura"])

# Ordeno las distancias de menor a mayor
np.sort(df.mahalanobis)

In [ ]:
p = 3  # ahora son 3 variables: Edad, Peso, Altura
umbral = np.sqrt(chi2.ppf(1 - alpha, df=p))
umbral

In [ ]:
# Observaciones anómalas
df[df["mahalanobis"] > umbral]

In [ ]:
# Gráfico 3D
es_outlier = df["mahalanobis"] > umbral

fig = plt.figure(figsize=(12, 12))
ax = fig.add_subplot(projection="3d")

color = [f"C{n}" for n in (es_outlier * 1)]
ax.scatter(df.Peso, df.Edad, df.Altura, c=color)
ax.set_xlabel("Peso")
ax.set_ylabel("Edad")
ax.set_zlabel("Altura")
plt.title("Dispersograma Peso Edad Altura")
plt.show()

¿Qué podemos hacer con estos valores anómalos que detectamos?

Podrían resultar de interés si, por ejemplo, queremos estudiar la relación de estas variables con ciertos aspectos de la salud.

In [ ]:
# Calculo el Índice de masa corporal
df["IMC"] = np.round(df.Peso / (df.Altura / 100) ** 2, 2)

# Observaciones anómalas
df[df["mahalanobis"] > umbral]

Una de las observaciones anómalas corresponde a una persona con sobrepeso, y otra a una persona con bajo peso.

### Nota: el problema del *masking* y una alternativa robusta

Mahalanobis "clásico" estima $\mu$ y $\Sigma$ usando **todos** los datos, incluyendo los propios outliers. Si hay varios outliers similares entre sí, pueden inflar la covarianza estimada de tal forma que se "escondan" (masking) — su distancia termina pareciendo normal porque la elipse de referencia se agrandó para incluirlos.

Una alternativa más robusta es estimar $\mu$ y $\Sigma$ con el **Minimum Covariance Determinant (MCD)**, que busca el subconjunto de observaciones cuya covarianza tiene determinante mínimo, ignorando a los outliers en la propia estimación:

In [ ]:
from sklearn.covariance import MinCovDet

cols_mcd = ["Edad", "Peso", "Altura"]
mcd = MinCovDet(random_state=0).fit(df[cols_mcd])

# Distancia de Mahalanobis robusta (usa mu y Sigma estimados por MCD)
df["mahalanobis_robusto"] = np.sqrt(mcd.mahalanobis(df[cols_mcd]))

df[["mahalanobis", "mahalanobis_robusto"]].describe()

**TODO:** Comparen `mahalanobis` vs `mahalanobis_robusto` con el mismo umbral chi-cuadrado. ¿Cambia el conjunto de observaciones detectadas como anómalas?

## Análisis Multivariado - Isolation Forest

Ahora vamos a leer el conjunto de datos de iris usando `load_iris`.

El objetivo es verificar las anomalías en este conjunto de datos, que tiene cuatro características por muestra: longitud y ancho de sépalos y pétalos.

Estas características serán evaluadas por el algoritmo Isolation Forest para determinar si las observaciones son anómalas o no.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.ensemble import IsolationForest

# Cargo el dataset
data = load_iris(as_frame=True)
X, y = data.data, data.target
df = data.frame
df.head()

In [ ]:
# Renombro las columnas
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "target"]
X.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

Vamos a definir el modelo. Hay hiperparámetros relevantes para definir:

* `contamination`: proporción esperada de anomalías en el conjunto de datos. Acá la fijamos en 0.05.
* `max_samples`: número máximo de muestras a considerar. Usaremos todas.
* `max_features`: número máximo de características a considerar durante el entrenamiento. Usaremos las cuatro.
* `n_estimators`: cantidad de árboles de aislamiento. Usaremos 100.

**Importante — algo que no se discute en la versión original:** en un caso real casi nunca conocemos la proporción real de anomalías, así que `contamination=0.05` es una suposición, no un dato. Por eso conviene mirar también el score continuo (`decision_function`) en vez de depender únicamente de la etiqueta binaria que produce ese contamination arbitrario — el score nos deja elegir el punto de corte según el contexto del problema, e incluso reevaluarlo sin reentrenar el modelo.

In [ ]:
# Creo el modelo
iforest = IsolationForest(
    n_estimators=100,
    max_samples="auto",
    contamination=0.05,
    max_features=4,
    bootstrap=False,
    n_jobs=-1,
    random_state=1,
)

Después de definir el modelo, lo ajustamos a los datos y obtenemos las etiquetas para X mediante `fit_predict`.

In [ ]:
# Entreno y genero la predicción
pred = iforest.fit_predict(X)

Podemos obtener el score de anomalía con `decision_function`. Cuando la etiqueta es -1 tenemos anomalías; si es 1, observaciones normales.

In [ ]:
# Almaceno scores y etiquetas
df["scores"] = iforest.decision_function(X)
df["outlier_label"] = pred

# Observaciones anómalas
df[df.outlier_label == -1]

In [ ]:
# Cantidad de obs. anómalas
df.outlier_label.value_counts()

In [ ]:
# Distribución del score continuo: sirve para elegir un punto de corte
# distinto al que impone `contamination`, sin reentrenar el modelo.
plt.hist(df["scores"], bins=20)
plt.axvline(0, color="red", linestyle="--", label="corte de decision_function (score=0)")
plt.title("Distribución del score de anomalía (Isolation Forest)")
plt.xlabel("score")
plt.legend()
plt.show()

Obtuvimos 8 muestras anómalas (con `contamination=0.05`), visualicemos las mismas en un dispersograma.

**TODO:** Graficar un pairplot con todas las variables resaltando los outliers detectados.

In [ ]:
# Grafico dispersograma
color = [f"C{n+1}" for n in (df["outlier_label"].values)]

sns.scatterplot(x=df["sepal_width"], y=df["petal_length"], hue=color)
plt.title("Diagrama de Dispersión sepal_width vs petal_length")
plt.show()

También podemos visualizar uno de los 100 estimadores. En este caso seleccionamos el quinto árbol.

**TODO:** Modificar el parámetro `max_depth` y visualizar nuevamente.

*(Nota: esta celda requiere `graphviz` instalado en el entorno de Colab.)*

In [ ]:
# Selecciono el árbol
estimator = iforest.estimators_[5]

# Grafico
plt.figure(figsize=(10, 10))
sk.tree.plot_tree(estimator, feature_names=data.feature_names, filled=True, max_depth=1)
plt.show()

In [ ]:
from sklearn.tree import export_graphviz
from subprocess import call
from IPython.display import Image

estimator = iforest.estimators_[5]

export_graphviz(
    estimator,
    out_file="tree.dot",
    max_depth=5,
    feature_names=data.feature_names,
    special_characters=True,
    rounded=True,
    precision=2,
)
call(["dot", "-Tpng", "tree.dot", "-o", "tree.png", "-Gdpi=600"])
Image(filename="tree.png")

## Análisis Multivariado - LOF (Local Outlier Factor)

Vamos a intentar detectar outliers multivariados sobre el dataset iris utilizando LOF.

Trabajaremos sobre las variables `sepal_length` y `petal_length`.

**TODO:** Probar el algoritmo sobre otro conjunto de variables.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.neighbors import LocalOutlierFactor

# Cargo el dataset
data = load_iris(as_frame=True)
df = data.frame
df.head()

In [ ]:
# Renombro las columnas
df.columns = ["sepal_length", "sepal_width", "petal_length", "petal_width", "target"]

sns.scatterplot(y=df["sepal_length"], x=df["petal_length"], hue=df["target"])
plt.title("Diagrama de Dispersión sepal_length vs petal_length")
plt.show()

In [ ]:
# Conjunto de entrenamiento
X = df[["petal_length", "sepal_length"]].values

# Creo el clasificador
clf = LocalOutlierFactor(n_neighbors=20)

# Genero predicción
y_pred = clf.fit_predict(X)

# Guardo los scores (negative_outlier_factor_: cuanto más negativo, más anómalo)
df["scores"] = clf.negative_outlier_factor_

df.head()

Visualicemos los datos y sus scores en un diagrama de dispersión.

In [ ]:
# Calculo radio para plotear score
radius = (df.scores.max() - df.scores) / (df.scores.max() - df.scores.min())

# Grafico LOF
plt.figure(figsize=(10, 10))
plt.title("Local Outlier Factor (LOF)")
plt.scatter(
    df.petal_length.values,
    df.sepal_length.values,
    edgecolor="grey",
    s=30,
    label="datos",
    facecolors="none",
)
plt.scatter(
    df.petal_length.values,
    df.sepal_length.values,
    s=1300 * radius,
    edgecolors="red",
    facecolors="none",
    label="Outlier scores",
)
legend = plt.legend(loc="upper left")
legend.legend_handles[0].set_sizes([10])
legend.legend_handles[1].set_sizes([20])
plt.show()

### Resolución del TODO: fijar un umbral para LOF

`y_pred` ya nos da una etiqueta binaria (-1 / 1) basada en `contamination` por defecto, igual que antes con Isolation Forest. Pero también podemos fijar nuestro propio umbral sobre el score continuo (`negative_outlier_factor_`) — por ejemplo, usando un percentil de la distribución de scores, que es un enfoque más transparente que un `contamination` implícito:

In [ ]:
# Umbral por percentil: marco como outliers el 5% con peor (más negativo) score
percentil_umbral = 5
umbral_lof = np.percentile(df["scores"], percentil_umbral)

es_outlier_lof = df["scores"] < umbral_lof

plt.figure(figsize=(8, 8))
sns.scatterplot(x=df["petal_length"], y=df["sepal_length"], hue=es_outlier_lof, palette={True: "red", False: "grey"})
plt.title(f"LOF - outliers bajo el percentil {percentil_umbral} del score")
plt.legend(title="Outlier")
plt.show()

df[es_outlier_lof]

## Cierre: ¿qué método uso en cada caso?

| Método | Univariado / Multivariado | Supone normalidad | Sensible a correlación entre variables | Da un score continuo | Cuándo conviene |
|---|---|---|---|---|---|
| Boxplot / IQR | Univariado | No | No aplica | No (solo etiqueta) | Exploración rápida, variable por variable |
| Z-score | Univariado | Sí | No aplica | Sí | Datos aprox. normales, sin outliers extremos que distorsionen media/std |
| Z-score modificado | Univariado | No (usa mediana/MAD) | No aplica | Sí | Igual que z-score, pero más robusto ante outliers severos |
| Mahalanobis | Multivariado | Sí (para el umbral chi-cuadrado) | Sí, la tiene en cuenta explícitamente | Sí | Variables numéricas correlacionadas, buscás anomalías en la *combinación* de variables |
| Mahalanobis + MCD | Multivariado | Sí | Sí | Sí | Igual que Mahalanobis, pero cuando sospechás masking (varios outliers similares) |
| Isolation Forest | Multivariado | No | Sí (implícitamente) | Sí | Datasets grandes, no lineales, sin supuestos distribucionales |
| LOF | Multivariado | No | Sí (vía densidad local) | Sí | Anomalías locales (una región densa dentro de otra menos densa), no solo anomalías "globales" |

**TODO final:** sobre el dataset de iris, comparen qué observaciones detecta cada uno de los tres métodos multivariados (Mahalanobis, Isolation Forest, LOF). ¿Coinciden? ¿Por qué podrían no coincidir?